# Multi-Head Attention: From Scratch to Sklearn**What this notebook covers:**- Mathematical foundations of multi-head attention (MHA)- Implementing MHA from scratch using only NumPy- Building a text classifier with MHA on real BBC News data- Comparing with sklearn's TF-IDF + LogisticRegression baseline- Visualizing attention patterns and analyzing hyperparameters**Prerequisites:** Linear algebra, probability, basic NLP, Python/NumPy**Dataset:** BBC News Classification- Kaggle link: https://www.kaggle.com/datasets/alfathterry/bbc-full-text-document-classification- 2,225 documents across 5 categories: business, entertainment, politics, sport, tech- D. Greene and P. Cunningham. "Practical Solutions to the Problem of Diagonal Dominance in Kernel Document Clustering." ICML 2006.

In [ ]:
!pip install kagglehub -qimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport kagglehubimport re, os, warningsfrom collections import Counterfrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import classification_report, confusion_matrix, accuracy_scorefrom sklearn.pipeline import Pipelinenp.random.seed(42)warnings.filterwarnings('ignore')

## Part 1: Theory Recap**5 key mathematical ideas behind multi-head attention:**1. **Scaled dot-product attention**: $\text{Attention}(Q, K, V) = \text{softmax}(QK^\top / \sqrt{d_k})V$ — each token queries every other token for relevance, then aggregates their values weighted by attention scores.2. **Multi-head projection**: Input is linearly projected into $h$ independent $d_k$-dimensional subspaces. Attention runs separately in each subspace, allowing each head to specialize (e.g., syntax, semantics, long-range dependencies).3. **Why $\sqrt{d_k}$ scaling**: Dot products grow with dimension — $\mathbb{E}[q \cdot k] = d_k \cdot \sigma^2$. Without scaling, softmax saturates, producing near-one-hot vectors with vanishing gradients. Dividing by $\sqrt{d_k}$ keeps variance $\approx 1$.4. **Parameter count**: One MHA layer has $4 \cdot d_\text{model}^2$ parameters, independent of the number of heads $h$. Four $d_\text{model} \times d_\text{model}$ matrices: $W_Q, W_K, W_V, W_O$.5. **Permutation equivariance**: Self-attention treats inputs as an unordered set — shuffling tokens shuffles outputs identically. Positional encodings must be added to restore order information.

### Load and Explore the DatasetWe download the BBC News dataset via `kagglehub`. Each row contains a news article and its category label. We inspect the structure, check for nulls, and understand class distribution.

In [ ]:
path = kagglehub.dataset_download("alfathterry/bbc-full-text-document-classification")data_dir = os.path.join(path, "bbc")print(f"Dataset downloaded to {path}")texts, labels = [], []for category in sorted(os.listdir(data_dir)):    cat_path = os.path.join(data_dir, category)    if os.path.isdir(cat_path):        for fname in os.listdir(cat_path):            with open(os.path.join(cat_path, fname), 'r', encoding='latin1') as f:                texts.append(f.read())            labels.append(category)df = pd.DataFrame({'text': texts, 'category': labels})print(f"Loaded {len(df)} documents across {df['category'].nunique()} categories")print()print(df['category'].value_counts())print()print(df.head())

In [ ]:
categories = sorted(df['category'].unique())cat_to_idx = {c: i for i, c in enumerate(categories)}df['label'] = df['category'].map(cat_to_idx)print("Label mapping:")for cat, idx in cat_to_idx.items():    print(f"  {idx}: {cat}")print(f"\nDataFrame info:")print(df.info())print(f"\nNull check:\n{df.isnull().sum()}")print(f"\nText length stats (words):")df['word_count'] = df['text'].apply(lambda x: len(x.split()))print(df['word_count'].describe())

## Part 2: From Scratch ImplementationWe implement a complete multi-head attention text classifier using **only NumPy**. The architecture:1. **Token embedding**: Learnable lookup table mapping token IDs to dense vectors2. **Sinusoidal positional encoding**: Injects sequence order3. **Multi-head self-attention**: The core mechanism4. **Feed-forward network**: Two linear layers with ReLU5. **Layer normalization**: Stabilizes training6. **Global average pooling**: Aggregates over the sequence dimension7. **Classification head**: Linear projection to class logitsWe implement forward AND backward passes for every component.

In [ ]:
class MultiHeadAttentionClassifier:    def __init__(self, vocab_size, d_model=32, num_heads=4, num_classes=5,                 max_seq_len=200, lr=0.01, epochs=10, batch_size=32):        self.vocab_size = vocab_size        self.d_model = d_model        self.num_heads = num_heads        self.d_k = d_model // num_heads        self.num_classes = num_classes        self.max_seq_len = max_seq_len        self.lr = lr        self.epochs = epochs        self.batch_size = batch_size        self.eps = 1e-6        np.random.seed(42)        scale = np.sqrt(2.0 / d_model)        # Token embedding matrix        self.embedding = np.random.randn(vocab_size, d_model) * 0.01        # MHA projection matrices (Q, K, V, O)        self.W_Q = np.random.randn(d_model, d_model) * scale        self.W_K = np.random.randn(d_model, d_model) * scale        self.W_V = np.random.randn(d_model, d_model) * scale        self.W_O = np.random.randn(d_model, d_model) * scale        # Feed-forward network: d_model -> 4*d_model -> d_model        self.W_ff1 = np.random.randn(d_model, 4 * d_model) * scale        self.b_ff1 = np.zeros(4 * d_model)        self.W_ff2 = np.random.randn(4 * d_model, d_model) * np.sqrt(2.0 / (4 * d_model))        self.b_ff2 = np.zeros(d_model)        # Layer norm parameters        self.ln_gamma = np.ones(d_model)        self.ln_beta = np.zeros(d_model)        # Classification head        self.W_cls = np.random.randn(d_model, num_classes) * scale        self.b_cls = np.zeros(num_classes)        # Fixed sinusoidal positional encoding        self.pos_enc = self._build_pos_encoding(max_seq_len, d_model)    # --- Positional Encoding (sinusoidal, as in "Attention Is All You Need") ---    def _build_pos_encoding(self, max_len, d_model):        pe = np.zeros((max_len, d_model))        pos = np.arange(max_len)[:, np.newaxis]        div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))        pe[:, 0::2] = np.sin(pos * div)        pe[:, 1::2] = np.cos(pos * div)        return pe[np.newaxis, :, :]  # (1, max_len, d_model)    # --- Numerically stable softmax ---    def _softmax(self, x, axis=-1):        x -= np.max(x, axis=axis, keepdims=True)        exp_x = np.exp(x)        return exp_x / (np.sum(exp_x, axis=axis, keepdims=True) + self.eps)    # --- Layer Normalization ---    def _layer_norm(self, x, gamma, beta):        mean = np.mean(x, axis=-1, keepdims=True)        var = np.var(x, axis=-1, keepdims=True)        x_hat = (x - mean) / np.sqrt(var + self.eps)        return gamma * x_hat + beta, (x_hat, mean, var)    def _layer_norm_backward(self, d_y, gamma, cache):        # Standard LN backward: corrects for mean/var dependencies        x_hat, mean, var = cache        sigma = np.sqrt(var + self.eps)        d_y_hat = d_y * gamma        dx = (d_y_hat - np.mean(d_y_hat, axis=-1, keepdims=True)              - x_hat * np.mean(d_y_hat * x_hat, axis=-1, keepdims=True)) / sigma        d_gamma = np.sum(d_y * x_hat, axis=(0, 1))        d_beta = np.sum(d_y, axis=(0, 1))        return dx, d_gamma, d_beta    # --- Multi-Head Self-Attention forward ---    def _mha_forward(self, x):        B, T, D = x.shape        # Linear projections        Q = x @ self.W_Q        K = x @ self.W_K        V = x @ self.W_V        # Reshape to multi-head: (B, H, T, d_k)        Q = Q.reshape(B, T, self.num_heads, self.d_k).transpose(0, 2, 1, 3)        K = K.reshape(B, T, self.num_heads, self.d_k).transpose(0, 2, 1, 3)        V = V.reshape(B, T, self.num_heads, self.d_k).transpose(0, 2, 1, 3)        # Scaled dot-product attention        scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(self.d_k)        attn_weights = self._softmax(scores, axis=-1)        context = attn_weights @ V        # Concatenate heads and project        context_concat = context.transpose(0, 2, 1, 3).reshape(B, T, D)        out = context_concat @ self.W_O        cache = (x, Q, K, V, scores, attn_weights, context, context_concat)        return out, cache    # --- Multi-Head Self-Attention backward ---    def _mha_backward(self, d_out, cache):        x, Q, K, V, scores, attn_weights, context, context_concat = cache        B, T, D = x.shape        d_k = self.d_k        # Gradient w.r.t. W_O        d_W_O = context_concat.reshape(B * T, D).T @ d_out.reshape(B * T, D)        d_context_concat = d_out @ self.W_O.T        # Through concat + reshape        d_context = d_context_concat.reshape(B, T, self.num_heads, d_k).transpose(0, 2, 1, 3)        # Through weighted sum: context = attn_weights @ V        d_attn_weights = d_context @ V.transpose(0, 1, 3, 2)        d_V = attn_weights.transpose(0, 1, 3, 2) @ d_context        # Through softmax        d_scores = attn_weights * (d_attn_weights - np.sum(            attn_weights * d_attn_weights, axis=-1, keepdims=True))        # Through Q @ K^T / sqrt(d_k)        d_Q = d_scores @ K / np.sqrt(d_k)        d_K = d_scores.transpose(0, 1, 3, 2) @ Q / np.sqrt(d_k)        # Reshape back        d_Q = d_Q.transpose(0, 2, 1, 3).reshape(B, T, D)        d_K = d_K.transpose(0, 2, 1, 3).reshape(B, T, D)        d_V = d_V.transpose(0, 2, 1, 3).reshape(B, T, D)        # Gradients w.r.t. projection matrices        d_W_Q = x.reshape(B * T, D).T @ d_Q.reshape(B * T, D)        d_W_K = x.reshape(B * T, D).T @ d_K.reshape(B * T, D)        d_W_V = x.reshape(B * T, D).T @ d_V.reshape(B * T, D)        # Gradient w.r.t. input (through all three paths)        dx = (d_Q @ self.W_Q.T + d_K @ self.W_K.T + d_V @ self.W_V.T)        grads = {'W_Q': d_W_Q, 'W_K': d_W_K, 'W_V': d_W_V, 'W_O': d_W_O}        return dx, grads    # --- Feed-Forward Network ---    def _ffn_forward(self, x):        hidden = x @ self.W_ff1 + self.b_ff1        hidden_relu = np.maximum(hidden, 0)        out = hidden_relu @ self.W_ff2 + self.b_ff2        cache = (x, hidden, hidden_relu, out)        return out, cache    def _ffn_backward(self, d_out, cache):        x, hidden, hidden_relu, ffn_out = cache        d_W_ff2 = hidden_relu.reshape(-1, hidden_relu.shape[-1]).T @ d_out.reshape(-1, d_out.shape[-1])        d_b_ff2 = np.sum(d_out, axis=(0, 1))        d_hidden_relu = d_out @ self.W_ff2.T        d_hidden = d_hidden_relu * (hidden > 0).astype(float)        d_W_ff1 = x.reshape(-1, x.shape[-1]).T @ d_hidden.reshape(-1, d_hidden.shape[-1])        d_b_ff1 = np.sum(d_hidden, axis=(0, 1))        dx = d_hidden @ self.W_ff1.T        grads = {'W_ff1': d_W_ff1, 'b_ff1': d_b_ff1, 'W_ff2': d_W_ff2, 'b_ff2': d_b_ff2}        return dx, grads    # --- Full forward ---    def forward(self, x, return_cache=True):        emb = self.embedding[x]        emb_pe = emb + self.pos_enc[:, :x.shape[1], :]        attn_out, mha_cache = self._mha_forward(emb_pe)        # LN1 residual block        ln1_in = emb_pe + attn_out        attn_residual, ln1_cache = self._layer_norm(ln1_in, self.ln_gamma, self.ln_beta)        # FFN residual block        ffn_out, ffn_cache = self._ffn_forward(attn_residual)        ln2_in = attn_residual + ffn_out        ffn_residual, ln2_cache = self._layer_norm(ln2_in, self.ln_gamma, self.ln_beta)        pooled = np.mean(ffn_residual, axis=1)        logits = pooled @ self.W_cls + self.b_cls        if return_cache:            cache = (emb_pe, mha_cache, attn_residual, ln1_cache,                     ffn_cache, ffn_residual, ln2_cache, pooled,                     ln1_in, ln2_in, ffn_out)            return logits, cache        return logits    # --- Full backward ---    def backward(self, logits, labels, cache):        emb_pe, mha_cache, attn_residual, ln1_cache = cache[:4]        ffn_cache, ffn_residual, ln2_cache, pooled = cache[4:8]        ln1_in, ln2_in, ffn_out = cache[8:]        B = logits.shape[0]        # Cross-entropy gradient        probs = self._softmax(logits, axis=-1)        d_logits = probs.copy()        d_logits[np.arange(B), labels] -= 1        d_logits /= B        d_W_cls = pooled.T @ d_logits        d_b_cls = np.sum(d_logits, axis=0)        d_pooled = d_logits @ self.W_cls.T        T = ffn_residual.shape[1]        d_ffn_residual = np.repeat(d_pooled[:, np.newaxis, :] / T, T, axis=1)        # LN2 backward        d_ffn_residual_ln, d_ln_gamma2, d_ln_beta2 = self._layer_norm_backward(            d_ffn_residual, self.ln_gamma, ln2_cache)        # FFN backward (grad flows through residual)        d_ffn, ffn_grads = self._ffn_backward(d_ffn_residual_ln, ffn_cache)        d_attn_residual = d_ffn_residual_ln + d_ffn        # LN1 backward        d_attn_residual_ln, d_ln_gamma1, d_ln_beta1 = self._layer_norm_backward(            d_attn_residual, self.ln_gamma, ln1_cache)        # MHA backward + residual        dx_mha, mha_grads = self._mha_backward(d_attn_residual_ln, mha_cache)        dx = d_attn_residual_ln + dx_mha  # total grad w.r.t. emb_pe (no PE grads)        grads = {            'W_cls': d_W_cls, 'b_cls': d_b_cls,            'W_Q': mha_grads['W_Q'], 'W_K': mha_grads['W_K'],            'W_V': mha_grads['W_V'], 'W_O': mha_grads['W_O'],            'W_ff1': ffn_grads['W_ff1'], 'b_ff1': ffn_grads['b_ff1'],            'W_ff2': ffn_grads['W_ff2'], 'b_ff2': ffn_grads['b_ff2'],            'ln_gamma': d_ln_gamma1 + d_ln_gamma2,            'ln_beta': d_ln_beta1 + d_ln_beta2,        }        return grads, dx    # --- Text preprocessing ---    def _tokenize_text(self, text):        text = text.lower()        text = re.sub(r'[^a-z0-9\s]', '', text)        return text.split()    def build_vocab(self, texts, max_vocab_size=5000):        counter = Counter()        for t in texts:            counter.update(self._tokenize_text(t))        vocab = ['<PAD>', '<UNK>'] + [w for w, _ in counter.most_common(max_vocab_size - 2)]        self.word2idx = {w: i for i, w in enumerate(vocab)}        self.idx2word = {i: w for w, i in self.word2idx.items()}        self.vocab_size = len(vocab)        print(f"Vocabulary size: {self.vocab_size}")    def encode(self, texts):        sequences = []        unk = self.word2idx.get('<UNK>')        pad = self.word2idx.get('<PAD>')        for text in texts:            tokens = self._tokenize_text(text)            seq = [self.word2idx.get(t, unk) for t in tokens[:self.max_seq_len]]            seq = seq + [pad] * (self.max_seq_len - len(seq))            sequences.append(seq)        return np.array(sequences, dtype=np.int64)    # --- Training loop ---    def fit(self, texts, y, texts_val=None, y_val=None, verbose=True):        np.random.seed(42)        self.build_vocab(texts)        X = self.encode(texts)        y = np.array(y)        X_val = self.encode(texts_val) if texts_val is not None else None        y_val = np.array(y_val) if y_val is not None else None        n = len(X)        history = {'train_loss': [], 'train_acc': [], 'val_acc': []}        for epoch in range(self.epochs):            perm = np.random.permutation(n)            X_shuf, y_shuf = X[perm], y[perm]            epoch_loss = 0.0            correct = 0            total = 0            for i in range(0, n, self.batch_size):                batch_X = X_shuf[i:i + self.batch_size]                batch_y = y_shuf[i:i + self.batch_size]                B = batch_X.shape[0]                # Forward                logits, cache = self.forward(batch_X, return_cache=True)                probs = self._softmax(logits, axis=-1)                loss = -np.mean(np.log(probs[np.arange(B), batch_y] + self.eps))                epoch_loss += loss * B                preds = np.argmax(logits, axis=1)                correct += np.sum(preds == batch_y)                total += B                # Backward                grads, dx = self.backward(logits, batch_y, cache)                # Embedding gradients (use np.add.at for efficiency)                d_embedding = np.zeros_like(self.embedding)                np.add.at(d_embedding, batch_X, dx)                # SGD update                self.embedding -= self.lr * d_embedding                for param in ['W_Q', 'W_K', 'W_V', 'W_O', 'W_ff1', 'W_ff2',                              'W_cls', 'ln_gamma', 'ln_beta']:                    setattr(self, param, getattr(self, param) - self.lr * grads[param])                for param in ['b_ff1', 'b_ff2', 'b_cls']:                    setattr(self, param, getattr(self, param) - self.lr * grads[param])            avg_loss = epoch_loss / total            train_acc = correct / total            history['train_loss'].append(avg_loss)            history['train_acc'].append(train_acc)            val_acc = 0.0            if X_val is not None and y_val is not None:                val_preds = self.predict(texts_val)                val_acc = np.mean(val_preds == y_val)                history['val_acc'].append(val_acc)            if verbose:                val_str = f" | val_acc: {val_acc:.4f}" if X_val is not None else ""                print(f"Epoch {epoch+1}/{self.epochs} - loss: {avg_loss:.4f} - train_acc: {train_acc:.4f}{val_str}")        return history    # --- Prediction ---    def predict(self, texts):        X = self.encode(texts)        logits = self.forward(X, return_cache=False)        return np.argmax(logits, axis=1)    def predict_proba(self, texts):        X = self.encode(texts)        logits = self.forward(X, return_cache=False)        return self._softmax(logits, axis=-1)

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(    df['text'].values, df['label'].values, test_size=0.2, random_state=42, stratify=df['label'].values)print(f"Train: {len(train_texts)} docs, Test: {len(test_texts)} docs")print("\nTraining Multi-Head Attention Classifier (from scratch, NumPy only)...")mha_model = MultiHeadAttentionClassifier(    vocab_size=5000, d_model=32, num_heads=4, num_classes=len(categories),    max_seq_len=200, lr=0.01, epochs=8, batch_size=32)history = mha_model.fit(train_texts, train_labels, test_texts, test_labels, verbose=True)train_preds = mha_model.predict(train_texts)test_preds = mha_model.predict(test_texts)train_acc = accuracy_score(train_labels, train_preds)test_acc = accuracy_score(test_labels, test_preds)print(f"\n{'='*50}")print(f"From-Scratch MHA Results:")print(f"{'='*50}")print(f"Train accuracy: {train_acc:.4f}")print(f"Test accuracy:  {test_acc:.4f}")print(f"\nClassification Report (Test):")print(classification_report(test_labels, test_preds, target_names=categories))

## Part 3: Sklearn ImplementationScikit-learn does not include multi-head attention; instead, we use **TF-IDF vectorization + Logistic Regression** as a powerful bag-of-words baseline. TF-IDF captures word importance via term frequency and inverse document frequency, while Logistic Regression provides a linear decision boundary.While MHA models *contextual relationships* between tokens, TF-IDF treats each document as a bag of independent words. Comparing the two reveals the value of attention-based modeling.

In [ ]:
print("Training sklearn TF-IDF + LogisticRegression pipeline...")sklearn_pipeline = Pipeline([    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),    ('clf', LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial'))])sklearn_pipeline.fit(train_texts, train_labels)sk_train_preds = sklearn_pipeline.predict(train_texts)sk_test_preds = sklearn_pipeline.predict(test_texts)sk_train_acc = accuracy_score(train_labels, sk_train_preds)sk_test_acc = accuracy_score(test_labels, sk_test_preds)print(f"\n{'='*50}")print(f"Sklearn (TF-IDF + LogisticRegression) Results:")print(f"{'='*50}")print(f"Train accuracy: {sk_train_acc:.4f}")print(f"Test accuracy:  {sk_test_acc:.4f}")print(f"\nClassification Report (Test):")print(classification_report(test_labels, sk_test_preds, target_names=categories))print(f"\n{'='*50}")print(f"Comparison Summary")print(f"{'='*50}")print(f"{'Model':<45} {'Train Acc':<12} {'Test Acc':<12}")print(f"{'-'*45} {'-'*12} {'-'*12}")print(f"{'From-Scratch MHA (NumPy)':<45} {train_acc:<12.4f} {test_acc:<12.4f}")print(f"{'Sklearn TF-IDF + LogisticRegression':<45} {sk_train_acc:<12.4f} {sk_test_acc:<12.4f}")

In [ ]:
sns.set_style("whitegrid")fig, axes = plt.subplots(2, 2, figsize=(14, 10))ax = axes[0, 0]ax.plot(history['train_loss'], 'b-o', linewidth=2, markersize=6)ax.set_title('Training Loss (From-Scratch MHA)', fontsize=13, fontweight='bold')ax.set_xlabel('Epoch')ax.set_ylabel('Cross-Entropy Loss')ax.grid(True, alpha=0.3)ax = axes[0, 1]ax.plot(history['train_acc'], 'b-o', linewidth=2, markersize=6, label='Train Acc')ax.plot(history['val_acc'], 'r-s', linewidth=2, markersize=6, label='Val Acc')ax.set_title('Accuracy Over Epochs (From-Scratch MHA)', fontsize=13, fontweight='bold')ax.set_xlabel('Epoch')ax.set_ylabel('Accuracy')ax.legend(fontsize=11)ax.grid(True, alpha=0.3)ax = axes[1, 0]cm_mha = confusion_matrix(test_labels, test_preds)sns.heatmap(cm_mha, annot=True, fmt='d', cmap='Blues', ax=ax,            xticklabels=categories, yticklabels=categories)ax.set_title('Confusion Matrix — MHA (Scratch)', fontsize=13, fontweight='bold')ax.set_xlabel('Predicted')ax.set_ylabel('True')ax = axes[1, 1]cm_sk = confusion_matrix(test_labels, sk_test_preds)sns.heatmap(cm_sk, annot=True, fmt='d', cmap='Greens', ax=ax,            xticklabels=categories, yticklabels=categories)ax.set_title('Confusion Matrix — Sklearn (TF-IDF + LR)', fontsize=13, fontweight='bold')ax.set_xlabel('Predicted')ax.set_ylabel('True')plt.tight_layout()plt.show()fig, ax = plt.subplots(figsize=(8, 5))models = ['From-Scratch MHA', 'Sklearn TF-IDF + LR']train_accs = [train_acc, sk_train_acc]test_accs = [test_acc, sk_test_acc]x = np.arange(len(models))width = 0.35bars1 = ax.bar(x - width/2, train_accs, width, label='Train Accuracy', color='steelblue', edgecolor='black')bars2 = ax.bar(x + width/2, test_accs, width, label='Test Accuracy', color='coral', edgecolor='black')ax.set_ylabel('Accuracy', fontsize=12)ax.set_title('Model Comparison', fontsize=14, fontweight='bold')ax.set_xticks(x)ax.set_xticklabels(models, fontsize=11)ax.legend(fontsize=11)ax.grid(axis='y', alpha=0.3)for bar in bars1:    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)for bar in bars2:    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)plt.tight_layout()plt.show()

## Part 4: Hyperparameter ExperimentsFor multi-head attention, the two most impactful hyperparameters are:1. **Number of heads ($h$)**: Controls how many independent attention subspaces the model learns. Too few heads limit expressivity; too many heads produce redundant/collapsed patterns.2. **Embedding dimension ($d_\text{model}$)**: The size of token representations and hidden dimension. Larger $d_\text{model}$ increases capacity and parameter count. Must be divisible by $h$.We vary both and measure test accuracy.

In [ ]:
print("=" * 60)print("Experiment 1: Varying Number of Heads")print("=" * 60)head_values = [2, 4, 8]head_results = []for h in head_values:    print(f"\nTraining with num_heads={h}...")    model = MultiHeadAttentionClassifier(        vocab_size=5000, d_model=32, num_heads=h, num_classes=len(categories),        max_seq_len=200, lr=0.01, epochs=6, batch_size=32    )    model.fit(train_texts, train_labels, verbose=False)    preds = model.predict(test_texts)    acc = accuracy_score(test_labels, preds)    head_results.append(acc)    print(f"  Test accuracy = {acc:.4f}")print("\n" + "=" * 60)print("Experiment 2: Varying Embedding Dimension")print("=" * 60)d_model_values = [16, 32, 64]d_model_results = []for d in d_model_values:    h = min(4, d // 2)    print(f"\nTraining with d_model={d}, num_heads={h}...")    model = MultiHeadAttentionClassifier(        vocab_size=5000, d_model=d, num_heads=h, num_classes=len(categories),        max_seq_len=200, lr=0.01, epochs=6, batch_size=32    )    model.fit(train_texts, train_labels, verbose=False)    preds = model.predict(test_texts)    acc = accuracy_score(test_labels, preds)    d_model_results.append(acc)    print(f"  Test accuracy = {acc:.4f}")fig, axes = plt.subplots(1, 2, figsize=(14, 5))ax = axes[0]ax.bar([str(h) for h in head_values], head_results, color='cornflowerblue', edgecolor='black', width=0.5)ax.set_xlabel('Number of Heads', fontsize=12)ax.set_ylabel('Test Accuracy', fontsize=12)ax.set_title('Effect of Number of Heads', fontsize=13, fontweight='bold')ax.set_ylim(0, 1)for i, v in enumerate(head_results):    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11)ax.grid(axis='y', alpha=0.3)ax = axes[1]ax.bar([str(d) for d in d_model_values], d_model_results, color='lightcoral', edgecolor='black', width=0.5)ax.set_xlabel('Embedding Dimension (d_model)', fontsize=12)ax.set_ylabel('Test Accuracy', fontsize=12)ax.set_title('Effect of Embedding Dimension', fontsize=13, fontweight='bold')ax.set_ylim(0, 1)for i, v in enumerate(d_model_results):    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11)ax.grid(axis='y', alpha=0.3)plt.tight_layout()plt.show()

## Part 5: Interview Corner**Q: Walk through the full forward pass of multi-head attention, including all tensor shapes and the parameter count.**---**Answer:**We start with an input sequence $X \in \mathbb{R}^{n \times d_{\text{model}}}$ where $n$ is the sequence length and $d_{\text{model}}$ is the embedding dimension.**Step 1 — Linear projections:**$$Q = X W_Q,\quad K = X W_K,\quad V = X W_V$$where $W_Q, W_K, W_V \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$. Each output: $\mathbb{R}^{n \times d_{\text{model}}}$.**Step 2 — Split into heads:**Reshape each of $Q, K, V$ from $(n, d_{\text{model}})$ to $(n, h, d_k)$, transpose to $(h, n, d_k)$, with $d_k = d_{\text{model}} / h$. Batched: $(B, h, n, d_k)$.**Step 3 — Scaled dot-product attention:**$$\text{scores} = Q K^\top \in \mathbb{R}^{h \times n \times n}$$Scale by $1/\sqrt{d_k}$.**Step 4 — Softmax:**$$\text{attn} = \text{softmax}(\text{scores}) \in \mathbb{R}^{h \times n \times n}$$Converts raw scores to a probability distribution over positions.**Step 5 — Weighted sum:**$$\text{context} = \text{attn} \cdot V \in \mathbb{R}^{h \times n \times d_k}$$Blends value vectors weighted by attention probabilities.**Step 6 — Concatenate heads:**Transpose back and reshape to $(n, h \cdot d_k) = (n, d_{\text{model}})$.**Step 7 — Output projection:**$$\text{out} = \text{context}_{\text{concat}} W_O,\quad W_O \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$$**Parameter count:**Four $d_{\text{model}} \times d_{\text{model}}$ matrices = $4 d_{\text{model}}^2$ parameters per MHA layer. This is **independent of $h$**. For $d_{\text{model}} = 512$, that's $4 \times 512^2 = 1,048,576$ parameters.

## Key Takeaways1. **Multi-head attention computes multiple independent attention functions in parallel** — each head projects the input into a different subspace and can specialize in different patterns (e.g., syntax, coreference, long-range dependencies).2. **Parameter count is $4 d_{\text{model}}^2$, independent of the number of heads** — heads share the $d_{\text{model}}$ budget via $d_k = d_{\text{model}} / h$. This is a clean design property to remember for interviews.3. **The $\sqrt{d_k}$ scaling factor is not optional** — without it, dot-product variances grow with dimension, softmax saturates, and gradients vanish. Most common "why" question asked about attention.4. **MHA is permutation-equivariant without positional encodings** — shuffling input tokens produces identically shuffled outputs. Sinusoidal, learned, or RoPE encodings are essential for sequence-order tasks.5. **From-scratch implementation reveals the exact tensor operations behind each equation** — building MHA with NumPy demystifies library abstractions like `nn.MultiheadAttention` and makes debugging and extension far easier.